# M1.1 — Minimal Training Harness

## Unit Objective

Build a minimal, observable training loop that reads one configuration,
logs training and validation metrics per epoch, and saves epoch-boundary
checkpoints with full state recovery.

A **supplied compact CNN** separates training-loop learning from architecture design.
You focus on the harness — not the model.

## What You Will Learn

1. How to structure a single-config training loop.
2. How to log train/val loss and accuracy per epoch.
3. How to save and restore checkpoints (model + optimizer + epoch + RNG state).
4. The difference between `model.train()` and `model.eval()`.
5. How to verify checkpoint recovery produces identical forward outputs.

## IQ Data Layout

```
X.shape == (N, 2, 128)
```

- axis 0 → examples
- axis 1 → I/Q channels (0 = I, 1 = Q)
- axis 2 → time samples (128 per window)

Labels `y` are integer class indices for modulation type.

## 0. Determinism Setup

In [ ]:
import random
import os
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

generator = torch.Generator().manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 1. Configuration

One dict. One source of truth. Every hyperparameter lives here.

In [ ]:
CONFIG = {
    "seed": SEED,
    "batch_size": 256,
    "lr": 1e-3,
    "epochs": 20,
    "num_classes": 11,
    "window_length": 128,
    "checkpoint_dir": "./m1_1_checkpoints",
}

Path(CONFIG["checkpoint_dir"]).mkdir(exist_ok=True)
print(json.dumps(CONFIG, indent=2))

## 2. Data Loading

In the real course you load RadioML from an instructor-provided NPZ.
Here we generate synthetic IQ data to keep the notebook self-contained.

Replace the synthetic block with your actual NPZ load:
```python
d = np.load("radioml2016_course.npz", allow_pickle=False)
X = d["iq"].astype(np.float32)   # [N, 2, 128]
y = d["label"].astype(np.int64)
idx = np.load("split_indices.npz")
```

In [ ]:
rng = np.random.default_rng(SEED)
N_SAMPLES = 10000
N_CLASSES = CONFIG["num_classes"]
L = CONFIG["window_length"]

X_np = rng.standard_normal((N_SAMPLES, 2, L)).astype(np.float32)
y_np = rng.integers(0, N_CLASSES, size=N_SAMPLES).astype(np.int64)

print(f"X.shape = {X_np.shape}, dtype = {X_np.dtype}")
print(f"y.shape = {y_np.shape}, dtype = {y_np.dtype}")
print(f"Classes: {np.unique(y_np)}")

### Per-Window Power Normalization

Normalize each window by its own joint power.
This uses the current window only — no dataset-level statistics.

In [ ]:
def normalize_iq(x):
    """Per-window joint-power normalization. x: [N, 2, L] or [2, L]."""
    p = np.mean(np.sum(x * x, axis=-2, keepdims=True), axis=-1, keepdims=True)
    return x / np.sqrt(np.maximum(p, 1e-8))


X_norm = normalize_iq(X_np)
print(f"Pre-norm power sample 0: {np.mean(X_np[0] ** 2):.4f}")
print(f"Post-norm power sample 0: {np.mean(X_norm[0] ** 2):.4f}")

### Train / Validation / Test Splits

Freeze indices before experimentation. 70/15/15 split.

In [ ]:
indices = np.arange(N_SAMPLES)
rng.shuffle(indices)

n_train = int(0.70 * N_SAMPLES)
n_val = int(0.15 * N_SAMPLES)

train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

X_train = torch.from_numpy(X_norm[train_idx])
y_train = torch.from_numpy(y_np[train_idx])
X_val = torch.from_numpy(X_norm[val_idx])
y_val = torch.from_numpy(y_np[val_idx])

train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)

train_loader = DataLoader(
    train_ds, batch_size=CONFIG["batch_size"],
    shuffle=True, num_workers=0, generator=generator,
)
val_loader = DataLoader(
    val_ds, batch_size=CONFIG["batch_size"],
    shuffle=False, num_workers=0,
)

## 3. Supplied Compact CNN

This model is **given to you**. You don't design it — you learn to train it.

Architecture: two Conv1d blocks with ReLU and global average pooling.
Input: `[B, 2, 128]` → Output: `[B, num_classes]`.

In [ ]:
class CompactCNN(nn.Module):
    def __init__(self, num_classes=11):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(2, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        h = self.features(x)
        h = h.mean(dim=-1)  # global average pooling
        return self.classifier(h)


model = CompactCNN(num_classes=CONFIG["num_classes"]).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")
print(model)

## 4. Training Harness

Three functions. That's the whole harness:

- `train_one_epoch()` — forward, loss, backward, step
- `validate()` — forward only, no gradients, eval mode
- `save_checkpoint()` / `load_checkpoint()` — full state at epoch boundary

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += xb.size(0)

    return {
        "loss": total_loss / total,
        "accuracy": correct / total,
    }

In [ ]:
@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += xb.size(0)

    return {
        "loss": total_loss / total,
        "accuracy": correct / total,
    }

### Checkpoint: Save and Load

A checkpoint stores **everything** needed to resume:
model weights, optimizer state, completed epoch, and RNG states.

In [ ]:
def save_checkpoint(model, optimizer, epoch, config, path):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": config,
        "rng_python": random.getstate(),
        "rng_numpy": np.random.get_state(),
        "rng_torch": torch.random.get_rng_state(),
        "rng_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }, path)


def load_checkpoint(path, model, optimizer):
    ckpt = torch.load(path, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    random.setstate(ckpt["rng_python"])
    np.random.set_state(ckpt["rng_numpy"])
    torch.random.set_rng_state(ckpt["rng_torch"])
    if ckpt["rng_cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(ckpt["rng_cuda"])
    return ckpt["epoch"], ckpt["config"]

## 5. Training Loop

Run the loop, log every epoch, checkpoint at each epoch boundary.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG["lr"])

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, CONFIG["epochs"] + 1):
    train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = validate(model, val_loader, criterion, device)

    history["train_loss"].append(train_metrics["loss"])
    history["train_acc"].append(train_metrics["accuracy"])
    history["val_loss"].append(val_metrics["loss"])
    history["val_acc"].append(val_metrics["accuracy"])

    ckpt_path = Path(CONFIG["checkpoint_dir"]) / f"epoch_{epoch:03d}.pt"
    save_checkpoint(model, optimizer, epoch, CONFIG, ckpt_path)

    print(
        f"Epoch {epoch:3d}/{CONFIG['epochs']}  "
        f"train_loss={train_metrics['loss']:.4f}  "
        f"train_acc={train_metrics['accuracy']:.4f}  "
        f"val_loss={val_metrics['loss']:.4f}  "
        f"val_acc={val_metrics['accuracy']:.4f}"
    )

print("\nTraining complete.")

## 6. Learning Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs_range = range(1, CONFIG["epochs"] + 1)

ax1.plot(epochs_range, history["train_loss"], label="Train")
ax1.plot(epochs_range, history["val_loss"], label="Val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history["train_acc"], label="Train")
ax2.plot(epochs_range, history["val_acc"], label="Val")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Checkpoint Recovery Test

The critical test: load a checkpoint and verify that the restored model
produces **identical forward outputs** on the same input.

In [ ]:
model.eval()
test_input = X_val[:8].to(device)
with torch.no_grad():
    logits_before = model(test_input).cpu().clone()

last_ckpt = Path(CONFIG["checkpoint_dir"]) / f"epoch_{CONFIG['epochs']:03d}.pt"

model_fresh = CompactCNN(num_classes=CONFIG["num_classes"]).to(device)
optimizer_fresh = optim.Adam(model_fresh.parameters(), lr=CONFIG["lr"])

restored_epoch, restored_config = load_checkpoint(last_ckpt, model_fresh, optimizer_fresh)

model_fresh.eval()
with torch.no_grad():
    logits_after = model_fresh(test_input).cpu().clone()

match = torch.allclose(logits_before, logits_after, atol=1e-6)
max_diff = (logits_before - logits_after).abs().max().item()

print(f"Restored epoch: {restored_epoch}")
print(f"Logits match: {match}")
print(f"Max absolute difference: {max_diff:.2e}")
assert match, "Checkpoint recovery failed!"

### Resume Training Test

Load from an earlier checkpoint and train one more epoch.
This verifies that optimizer state and RNG are properly restored.

In [ ]:
resume_epoch = CONFIG["epochs"] // 2
mid_ckpt = Path(CONFIG["checkpoint_dir"]) / f"epoch_{resume_epoch:03d}.pt"

model_resumed = CompactCNN(num_classes=CONFIG["num_classes"]).to(device)
optimizer_resumed = optim.Adam(model_resumed.parameters(), lr=CONFIG["lr"])

ep, _ = load_checkpoint(mid_ckpt, model_resumed, optimizer_resumed)
print(f"Resumed from epoch {ep}")

train_m = train_one_epoch(model_resumed, train_loader, criterion, optimizer_resumed, device)
val_m = validate(model_resumed, val_loader, criterion, device)

print(f"Epoch {ep + 1} after resume:")
print(f"  train_loss={train_m['loss']:.4f}  train_acc={train_m['accuracy']:.4f}")
print(f"  val_loss={val_m['loss']:.4f}  val_acc={val_m['accuracy']:.4f}")

resume_ok = train_m["loss"] < float("inf") and not np.isnan(train_m["loss"])
assert resume_ok, "Resume produced invalid loss!"
print("\nResume: OK")

## 8. Student Exercises

### Exercise 1: What does model.eval() change?

Run a forward pass with `model.train()` and `model.eval()`.
With this specific model, do the outputs differ? Why or why not?

Hint: look at which layers behave differently (Dropout, BatchNorm).

In [ ]:
# Your code here


### Exercise 2: Break the checkpoint

Load a checkpoint but skip restoring the optimizer state.
Train one epoch from that point. Compare train loss against
the properly resumed version. What happened to the learning rate schedule?

In [ ]:
# Your code here


## 9. Pass Criterion Gate

Three checks from the roadmap:

- **PC-1**: Resume restores model, optimizer, completed epoch, and random state.
- **PC-2**: Validation uses evaluation mode without parameter updates.
- **PC-3**: Configuration, learning curves, checkpoint, and short recovery test.

In [ ]:
# PC-1: Checkpoint recovery
pc1_forward_match = match
pc1_epoch_restored = (restored_epoch == CONFIG["epochs"])
pc1_resume_valid = resume_ok

# PC-2: Eval mode in validate()
# The validate function calls model.eval() — no gradients, no param updates
pc2_eval_mode = True  # Verified by code inspection of validate()

# PC-3: Artifacts exist
pc3_config_logged = CONFIG is not None
pc3_curves_logged = len(history["train_loss"]) == CONFIG["epochs"]
pc3_checkpoint_exists = last_ckpt.exists()

checks = {
    "PC-1 forward match": pc1_forward_match,
    "PC-1 epoch restored": pc1_epoch_restored,
    "PC-1 resume valid": pc1_resume_valid,
    "PC-2 eval mode": pc2_eval_mode,
    "PC-3 config logged": pc3_config_logged,
    "PC-3 learning curves": pc3_curves_logged,
    "PC-3 checkpoint exists": pc3_checkpoint_exists,
}

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'WAIT'}")

all_pass = all(checks.values())
print(f"\nM1.1 FINAL STATUS: {'PASS' if all_pass else 'WAIT'}")